# 🔬 Eksperimen 1: Cross-Project Flaky Test Prediction (Full Factorial 48 Pipelines)
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Dokumen Rencana**: `Eksperimen_Bima/Rencana_Penelitian_Flaky_Test.md`  
**Hardware Target**: NVIDIA GeForce RTX 4050 Laptop GPU (CUDA 13.4, Driver 617.42)  
**Skenario**: Leave-One-Project-Out Cross-Validation (LOPO-CV) lintas 24 proyek Java FlakeFlagger.

## 1. Verifikasi Lingkungan & GPU RTX 4050

In [ ]:
import sys
import os
from pathlib import Path

# Tambahkan folder src ke sys.path
src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.append(src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()

## 2. Ingestion & Eksplorasi Data (C1 vs C2)

In [ ]:
from data_loader import load_dataset

# Memuat dataset strategi C1 (Row-Level Drop)
df_c1, feat_c1, proj_c1 = load_dataset("C1")
print(f"C1: {len(df_c1)} baris, {len(proj_c1)} proyek, {df_c1['flaky'].sum()} flaky ({df_c1['flaky'].mean()*100:.2f}%)")

# Memuat dataset strategi C2 (Project-Level Drop)
df_c2, feat_c2, proj_c2 = load_dataset("C2")
print(f"C2: {len(df_c2)} baris, {len(proj_c2)} proyek, {df_c2['flaky'].sum()} flaky ({df_c2['flaky'].mean()*100:.2f}%)")

## 3. Eksekusi 48 Pipeline LOPO-CV (Leave-One-Project-Out)

In [ ]:
from lopo_runner import run_all_48_pipelines

# Menjalankan seluruh 48 matriks pipeline eksperimen
# Gunakan GPU RTX 4050 untuk XGBoost
df_summary = run_all_48_pipelines(output_dir="../results", use_gpu=True)
df_summary.head(10)

## 4. Uji Signifikansi Statistik (Friedman & Nemenyi Post-hoc)

In [ ]:
from statistical_tests import run_statistical_analysis

stat_results = run_statistical_analysis(
    raw_results_dir="../results/raw_predictions",
    metric_col="PR_AUC",
    output_file="../results/friedman_nemenyi_results.csv"
)
stat_results.head(10)

## 5. Visualisasi Publikasi Ilmiah (Boxplot & Heatmap)

In [ ]:
from visualizer import plot_rq_factor_boxplots, plot_pipeline_heatmap

summary_csv = "../results/summary_metrics_48_pipelines.csv"
plots_dir = "../results/plots"

plot_rq_factor_boxplots(summary_csv, plots_dir)
plot_pipeline_heatmap(summary_csv, plots_dir)
print("Visualisasi grafik selesai dibuat!")